# for optimal control lecture slides

In [ ]:
from pathlib import Path
import re
from collections import defaultdict

INPUT_DIR = Path(r"E:\OneDrive - MSFT\.master_data\25-26ws\oc\Lecture\_Lecture Slides")
OUTPUT_DIR = INPUT_DIR / "_merged"
OUTPUT_DIR.mkdir(exist_ok=True)

EXTS = {".ppt", ".pptx", ".pdf"}

def numeric_key(path):
    return tuple(int(x) for x in re.findall(r"\d+", path.stem))

def get_chapter_group(path):
    """
    02_Chapter_dynamic_programming.pdf -> 02_Chapter
    02_Chapter_dynamic_programming_VL1_annotated.pdf -> 02_Chapter
    """
    # 修改逻辑：按文件名开头的两位章节号分组。
    m = re.match(r"^(\d{2})_Chapter", path.stem, re.IGNORECASE)
    if not m:
        return None
    return f"{m.group(1)}_Chapter"

def count_pdf_pages(path):
    try:
        from pypdf import PdfReader
    except ImportError:
        raise ImportError("请先运行：pip install pypdf")

    return len(PdfReader(str(path)).pages)

def merge_pdfs(files, output_path):
    try:
        from pypdf import PdfWriter
    except ImportError:
        raise ImportError("请先运行：pip install pypdf")

    writer = PdfWriter()
    file_pages = []

    for f in files:
        n = count_pdf_pages(f)
        file_pages.append((f, n))
        writer.append(str(f))

    with open(output_path, "wb") as out:
        writer.write(out)

    total_pages = count_pdf_pages(output_path)
    return file_pages, total_pages

def merge_ppts(files, output_path):
    try:
        import win32com.client as win32
    except ImportError:
        raise ImportError("请先运行：pip install pywin32。该方法需要 Windows + PowerPoint。")

    app = win32.Dispatch("PowerPoint.Application")
    app.Visible = True

    merged = app.Presentations.Add()
    file_pages = []

    try:
        while merged.Slides.Count > 0:
            merged.Slides(1).Delete()

        for f in files:
            src = app.Presentations.Open(str(f.resolve()), ReadOnly=True, WithWindow=False)
            n = src.Slides.Count
            file_pages.append((f, n))
            src.Close()

            if n > 0:
                merged.Slides.InsertFromFile(str(f.resolve()), merged.Slides.Count, 1, n)

        total_pages = merged.Slides.Count
        merged.SaveAs(str(output_path.resolve()))

    finally:
        merged.Close()
        app.Quit()

    return file_pages, total_pages

files = [
    p for p in INPUT_DIR.rglob("*")
    if p.is_file()
    and p.suffix.lower() in EXTS
    and not p.name.startswith("~$")
    and "_merged" not in p.parts
]

groups = defaultdict(list)

for f in files:
    group = get_chapter_group(f)
    if group:
        groups[group].append(f)

for group, fs in sorted(groups.items(), key=lambda x: numeric_key(Path(x[0]))):
    fs = sorted(fs, key=numeric_key)

    print("\n" + "=" * 80)
    print(f"分组：{group}")
    print(f"合并前文件数：{len(fs)}")

    ppt_files = [f for f in fs if f.suffix.lower() in {".ppt", ".pptx"}]
    pdf_files = [f for f in fs if f.suffix.lower() == ".pdf"]

    if ppt_files:
        out = OUTPUT_DIR / f"{group}.pptx"
        file_pages, total_pages = merge_ppts(ppt_files, out)

        print("\nPPT 文件：")
        for f, n in file_pages:
            print(f"  {f.name} -> {n} 页")
        print(f"合并后文件：{out}")
        print(f"合并后总页数：{total_pages}")

    if pdf_files:
        out = OUTPUT_DIR / f"{group}.pdf"
        file_pages, total_pages = merge_pdfs(pdf_files, out)

        print("\nPDF 文件：")
        for f, n in file_pages:
            print(f"  {f.name} -> {n} 页")
        print(f"合并后文件：{out}")
        print(f"合并后总页数：{total_pages}")

# Merge all ODS exams

In [5]:
from pathlib import Path
import re
from pypdf import PdfReader, PdfWriter

# 只需修改这个目录；本单元可独立运行。
EXAM_DIR = Path(r"E:\OneDrive - MSFT\.master_data\25-26ws\oc\Supplementary\Recent Exam Tasks n Relevant Topics Only")
EXAM_OUTPUT = EXAM_DIR / "_merged" / "ODS_Exams.pdf"

def relative_text(path):
    return " ".join(path.relative_to(EXAM_DIR).parts).lower()

def exam_metadata(path):
    text = relative_text(path)
    years = [int(x) for x in re.findall(r"\b(?:19|20)\d{2}\b", text)]
    year = min(years, default=9999)
    term = 0 if "summer" in text else 1 if "winter" in text else 2
    solution = int("solution" in text)  # 同场考试：试卷在前，答案在后。
    return text, years, year, term, solution

def exam_sort_key(path):
    text, _, year, term, solution = exam_metadata(path)
    numbers = tuple(int(x) for x in re.findall(r"\d+", path.stem))
    return year, term, solution, numbers, text

def exam_title(path):
    _, years, _, term, solution = exam_metadata(path)
    period = "/".join([str(years[0]), *(str(y)[-2:] for y in years[1:])]) if years else "Unknown year"
    term_name = ("Summer", "Winter", "Exam")[term]
    kind = "Solution" if solution else "Exam"
    return f"{period} {term_name} — {kind}"

def find_exam_pdfs():
    files = (p for p in EXAM_DIR.rglob("*") if p.is_file() and p.suffix.lower() == ".pdf")
    return sorted((p for p in files if "exam" in relative_text(p) and "_merged" not in p.parts), key=exam_sort_key)

def merge_pdfs(files):
    writer = PdfWriter()
    total_pages = 0
    for path in files:
        pages = len(PdfReader(str(path)).pages)
        # outline_item 添加总目录；原 PDF 自带的目录会保留在该项下面。
        writer.append(str(path), outline_item=exam_title(path))
        total_pages += pages
        print(f"{path.relative_to(EXAM_DIR)} -> {pages} 页")

    EXAM_OUTPUT.parent.mkdir(parents=True, exist_ok=True)
    with EXAM_OUTPUT.open("wb") as output:
        writer.write(output)
    return total_pages

exam_files = find_exam_pdfs()
if not exam_files:
    raise FileNotFoundError(f"未找到考试 PDF：{EXAM_DIR}")

total_pages = merge_pdfs(exam_files)
print(f"合并完成：{EXAM_OUTPUT}")
print(f"文件数：{len(exam_files)}，总页数：{total_pages}")

ODS_Exam_WT20-21.pdf -> 6 页
ODS_Exam_ST21.pdf -> 7 页
ODS_Exam_WT21-22.pdf -> 6 页
ODS_Exam_ST22.pdf -> 6 页
ODS_Exam_WT22-23.pdf -> 7 页
ODS_Exam_ST23.pdf -> 7 页
ODS_Exam_WT23-24.pdf -> 8 页
ODS_Exam_ST24.pdf -> 8 页
ODS_Exam_WT24-25.pdf -> 9 页
ODS_Exam_ST25.pdf -> 8 页
合并完成：E:\OneDrive - MSFT\.master_data\25-26ws\oc\Supplementary\Recent Exam Tasks n Relevant Topics Only\_merged\ODS_Exams.pdf
文件数：10，总页数：72


拆分大页码，为200

In [ ]:
from pathlib import Path

try:
    from pypdf import PdfReader, PdfWriter
except ImportError:
    raise ImportError("请先运行：pip install pypdf")

PDF_PATH = Path(r"E:\OneDrive - MSFT\.master_data\25-26ws\hscd\2311620 n Hardware-Software Co-Design (WS 25_26)\Lecture slides\_merged\HSC_2.pdf")
MAX_PAGES = 200

reader = PdfReader(str(PDF_PATH))
total_pages = len(reader.pages)

output_dir = PDF_PATH.parent / f"{PDF_PATH.stem}_split_max_{MAX_PAGES}"
output_dir.mkdir(exist_ok=True)

print("=" * 80)
print(f"输入文件：{PDF_PATH}")
print(f"原始总页数：{total_pages}")
print(f"每个文件最多页数：{MAX_PAGES}")
print("=" * 80)

part_files = []

for start in range(0, total_pages, MAX_PAGES):
    end = min(start + MAX_PAGES, total_pages)
    part_id = start // MAX_PAGES + 1

    writer = PdfWriter()

    for page_idx in range(start, end):
        writer.add_page(reader.pages[page_idx])

    out_path = output_dir / f"{PDF_PATH.stem}_part{part_id:02d}_p{start + 1:03d}-{end:03d}.pdf"

    with open(out_path, "wb") as f:
        writer.write(f)

    part_pages = end - start
    part_files.append((out_path, part_pages))

    print(f"第 {part_id} 部分：第 {start + 1} 页到第 {end} 页，共 {part_pages} 页")
    print(f"输出文件：{out_path}")

print("=" * 80)
print(f"拆分后文件数：{len(part_files)}")
print(f"拆分后总页数：{sum(n for _, n in part_files)}")

if sum(n for _, n in part_files) == total_pages:
    print("检查通过：拆分前后总页数一致")
else:
    print("警告：拆分前后总页数不一致")

# for optimal control exercise

In [ ]:
from pathlib import Path
import re
from collections import defaultdict

INPUT_DIR = Path(r"E:\OneDrive - MSFT\.master_data\25-26ws\oc\Exercise_NEW")
OUTPUT_DIR = INPUT_DIR / "_merged"
OUTPUT_DIR.mkdir(exist_ok=True)

EXTS = {".ppt", ".pptx", ".pdf"}

def get_ex_group(path):
    """
    Exercise 1/OC_Exercise_1_Tasks_WT25-26.pdf -> EX01
    QaA/OC_QA_WT25-26_slides.pdf -> QA
    """
    # 修改逻辑：按父文件夹分组，避免 Exercise 4 中编号写成 1 的文件被分错。
    m = re.fullmatch(r"Exercise\s*0*(\d+)", path.parent.name, re.IGNORECASE)
    if m:
        return f"EX{int(m.group(1)):02d}"
    return "QA" if path.parent.name.lower() == "qaa" else None

def exercise_sort_key(path):
    name = path.stem.lower()

    if "presentation" in name or "slides" in name or "annotated" in name:
        role = 0
    elif "tasks-solution" in name or "solution" in name:
        role = 2
    elif "tasks" in name:
        role = 1
    else:
        role = 9

    nums = tuple(int(x) for x in re.findall(r"\d+", name))
    return nums, role, name

def count_pdf_pages(path):
    try:
        from pypdf import PdfReader
    except ImportError:
        raise ImportError("请先运行：pip install pypdf")

    return len(PdfReader(str(path)).pages)

def merge_pdfs(files, output_path):
    try:
        from pypdf import PdfWriter
    except ImportError:
        raise ImportError("请先运行：pip install pypdf")

    writer = PdfWriter()
    file_pages = []

    for f in files:
        n = count_pdf_pages(f)
        file_pages.append((f, n))
        writer.append(str(f))

    with open(output_path, "wb") as out:
        writer.write(out)

    total_pages = count_pdf_pages(output_path)
    return file_pages, total_pages

def merge_ppts(files, output_path):
    try:
        import win32com.client as win32
    except ImportError:
        raise ImportError("请先运行：pip install pywin32。该方法需要 Windows + PowerPoint。")

    app = win32.Dispatch("PowerPoint.Application")
    app.Visible = True

    merged = app.Presentations.Add()
    file_pages = []

    try:
        while merged.Slides.Count > 0:
            merged.Slides(1).Delete()

        for f in files:
            src = app.Presentations.Open(str(f.resolve()), ReadOnly=True, WithWindow=False)
            n = src.Slides.Count
            file_pages.append((f, n))
            src.Close()

            if n > 0:
                merged.Slides.InsertFromFile(str(f.resolve()), merged.Slides.Count, 1, n)

        total_pages = merged.Slides.Count
        merged.SaveAs(str(output_path.resolve()))

    finally:
        merged.Close()
        app.Quit()

    return file_pages, total_pages

files = [
    p for p in INPUT_DIR.rglob("*")
    if p.is_file()
    and p.suffix.lower() in EXTS
    and not p.name.startswith("~$")
    and "_merged" not in p.parts
]

groups = defaultdict(list)

for f in files:
    group = get_ex_group(f)
    if group:
        groups[group].append(f)

for group, fs in sorted(groups.items()):
    fs = sorted(fs, key=exercise_sort_key)

    print("\n" + "=" * 80)
    print(f"分组：{group}")
    print(f"合并前文件数：{len(fs)}")

    ppt_files = [f for f in fs if f.suffix.lower() in {".ppt", ".pptx"}]
    pdf_files = [f for f in fs if f.suffix.lower() == ".pdf"]

    if ppt_files:
        out = OUTPUT_DIR / f"{group}.pptx"
        file_pages, total_pages = merge_ppts(ppt_files, out)

        print("\nPPT 文件：")
        for f, n in file_pages:
            print(f"  {f.name} -> {n} 页")
        print(f"合并后文件：{out}")
        print(f"合并后总页数：{total_pages}")

    if pdf_files:
        out = OUTPUT_DIR / f"{group}.pdf"
        file_pages, total_pages = merge_pdfs(pdf_files, out)

        print("\nPDF 文件：")
        for f, n in file_pages:
            print(f"  {f.name} -> {n} 页")
        print(f"合并后文件：{out}")
        print(f"合并后总页数：{total_pages}")

裁剪不合适的翻译文件

In [ ]:
from pathlib import Path
from pypdf import PdfReader, PdfWriter

# 只改这里
PDF_PATH = Path(r"E:\OneDrive - MSFT\.master_data\25-26ws\hscd\2311620 n Hardware-Software Co-Design (WS 25_26)\Lecture slides\_merged\_HSC_2_part03_p401-547.pdf")

# 裁掉多少，单位 mm
CROP_LEFT_MM = 0
CROP_TOP_MM = 0
CROP_RIGHT_MM = 51.5
CROP_BOTTOM_MM = 25

def mm_to_pt(mm):
    return mm * 72 / 25.4

reader = PdfReader(str(PDF_PATH))
writer = PdfWriter()

crop_left = mm_to_pt(CROP_LEFT_MM)
crop_top = mm_to_pt(CROP_TOP_MM)
crop_right = mm_to_pt(CROP_RIGHT_MM)
crop_bottom = mm_to_pt(CROP_BOTTOM_MM)

for page in reader.pages:
    page_width = float(page.mediabox.width)
    page_height = float(page.mediabox.height)

    # PDF 坐标原点在左下角
    new_left = crop_left
    new_bottom = crop_bottom
    new_right = page_width - crop_right
    new_top = page_height - crop_top

    page.cropbox.lower_left = (new_left, new_bottom)
    page.cropbox.upper_right = (new_right, new_top)

    writer.add_page(page)

out_path = PDF_PATH.with_name(PDF_PATH.stem + "_cropped.pdf")

with open(out_path, "wb") as f:
    writer.write(f)

print("=" * 80)
print(f"原始文件：{PDF_PATH}")
print(f"原始页数：{len(reader.pages)}")
print(f"裁掉左边：{CROP_LEFT_MM} mm")
print(f"裁掉上边：{CROP_TOP_MM} mm")
print(f"裁掉右边：{CROP_RIGHT_MM} mm")
print(f"裁掉下边：{CROP_BOTTOM_MM} mm")
print(f"输出文件：{out_path}")
print("=" * 80)

翻译？


In [ ]:
from pathlib import Path
import subprocess
import tempfile
import shutil

# 只改这里
FOLDER = Path(r"E:\OneDrive - MSFT\.master_data\25-26ws\hscd\2311620 n Hardware-Software Co-Design (WS 25_26)\Lecture slides\_merged\test")

# 翻译设置
SOURCE_LANG = "en"
TARGET_LANG = "zh"

# 默认用 pdf2zh 自带默认翻译服务
# 想指定服务可以写 "google", "deepl", "openai" 等
SERVICE = None

# 已经存在 -原文件名.pdf 时是否覆盖
OVERWRITE = False

pdf2zh_cmd = shutil.which("pdf2zh")
if pdf2zh_cmd is None:
    raise RuntimeError("没有找到 pdf2zh。请先运行：pip install pdf2zh")

pdf_files = sorted([
    p for p in FOLDER.glob("*.pdf")
    if p.is_file()
    and not p.name.startswith("-")
    and not p.name.startswith("~$")
])

print("=" * 80)
print(f"待翻译文件夹：{FOLDER}")
print(f"待翻译 PDF 数量：{len(pdf_files)}")
print("=" * 80)

for pdf_path in pdf_files:
    out_path = pdf_path.with_name("-" + pdf_path.name)

    if out_path.exists() and not OVERWRITE:
        print(f"跳过，已存在：{out_path.name}")
        continue

    print("\n" + "-" * 80)
    print(f"开始翻译：{pdf_path.name}")

    with tempfile.TemporaryDirectory() as tmpdir:
        tmpdir = Path(tmpdir)
        tmp_input = tmpdir / pdf_path.name
        shutil.copy2(pdf_path, tmp_input)

        cmd = [
            pdf2zh_cmd,
            str(tmp_input),
            "-li", SOURCE_LANG,
            "-lo", TARGET_LANG,
        ]

        if SERVICE:
            cmd += ["-s", SERVICE]

        result = subprocess.run(
            cmd,
            cwd=tmpdir,
            text=True,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT
        )

        print(result.stdout[-2000:])

        if result.returncode != 0:
            print(f"失败：{pdf_path.name}")
            continue

        # 优先找双语 dual 文件
        candidates = list(tmpdir.glob(f"{pdf_path.stem}*dual*.pdf"))

        # 兼容不同版本输出名
        if not candidates:
            candidates = list(tmpdir.glob(f"{pdf_path.stem}*zh*.pdf"))
        if not candidates:
            candidates = list(tmpdir.glob(f"{pdf_path.stem}*mono*.pdf"))

        if not candidates:
            print(f"没有找到翻译输出文件：{pdf_path.name}")
            continue

        translated_pdf = candidates[0]

        if out_path.exists() and OVERWRITE:
            out_path.unlink()

        shutil.move(str(translated_pdf), str(out_path))

        print(f"完成：{out_path.name}")

print("\n" + "=" * 80)
print("全部处理完成")
print("=" * 80)